# Checkworthy Detection Tests
Tests various checkworthy detection methods against all `article*.json` files in the directory.

In [37]:
import sys
import os
import glob
import json
import logging
from typing import List, Dict

# Setup Paths
project_root = os.path.abspath('../../..') 
if project_root not in sys.path:
    sys.path.append(project_root)

# Import Backend Models
try:
    from common.models.api.redis_models import (
        Article, NLPResult, NLPOptions, Claim, SentenceScore
    )
    from microservices.nlp.models.base import NLPComponent
    from microservices.nlp.components.preprocess import Preprocessor
    print("Successfully loaded backend data structures.")
except ImportError as e:
    print(f"Import Failed: {e}")

# Logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("CheckworthyTest")

Successfully loaded backend data structures.


## Load All Articles
Load all `article*.json` files from the test directory to create a comprehensive test pool.

In [38]:
# Find all article JSON files
json_files = sorted(glob.glob("article*.json"))
print(f"Found {len(json_files)} article files: {json_files}")

# Load all articles into a list
articles = []
article_metadata = []

for fpath in json_files:
    try:
        with open(fpath, 'r') as f:
            data = json.load(f)
        
        # Create Article object
        article = Article(
            title=data.get('article_title', 'Unknown Title'),
            text=data.get('article_text', ''),
            link=data.get('article_url', ''),
            summary=data.get('article_summary', '')
        )
        
        articles.append(article)
        article_metadata.append({
            'filename': fpath,
            'title': article.title,
            'text_length': len(article.text),
            'has_summary': bool(article.summary)
        })
        
        print(f"✓ Loaded: {fpath} - {article.title[:60]}...")
        
    except Exception as e:
        print(f"✗ Error loading {fpath}: {e}")

print(f"\n{'='*60}")
print(f"Successfully loaded {len(articles)} articles")
print(f"{'='*60}")

# Display summary statistics
if article_metadata:
    total_chars = sum(m['text_length'] for m in article_metadata)
    avg_chars = total_chars // len(article_metadata)
    print(f"\nTotal text: {total_chars:,} characters")
    print(f"Average article length: {avg_chars:,} characters")
    print(f"Articles with summaries: {sum(m['has_summary'] for m in article_metadata)}")

Found 6 article files: ['article.json', 'article1.json', 'article2.json', 'article3.json', 'article4.json', 'article5.json']
✓ Loaded: article.json - title...
✓ Loaded: article1.json - What could happen if the US strikes Iran? Here are seven sce...
✓ Loaded: article2.json - What next for Venezuela? What leaders and experts said at Da...
✓ Loaded: article3.json - FBI raids Georgia election office over 2020 voter fraud clai...
✓ Loaded: article4.json - AI 'slop' is transforming social media - and a backlash is b...
✓ Loaded: article5.json - New files deepen a critical mystery about those who partied ...

Successfully loaded 6 articles

Total text: 50,969 characters
Average article length: 8,494 characters
Articles with summaries: 6


## Display Article Details
Quick overview of each loaded article.

In [39]:
# Display detailed information for each article
for idx, (article, meta) in enumerate(zip(articles, article_metadata), 1):
    print(f"\n{'='*60}")
    print(f"Article {idx}: {meta['filename']}")
    print(f"{'='*60}")
    print(f"Title: {article.title}")
    print(f"Link: {article.link}")
    print(f"Text Length: {meta['text_length']:,} characters")
    
    # Show first 200 characters of text
    text_preview = article.text[:200].replace('\n', ' ')
    print(f"Text Preview: {text_preview}...")
    
    if article.summary:
        summary_preview = article.summary[:150].replace('\n', ' ')
        print(f"Summary: {summary_preview}...")


Article 1: article.json
Title: title
Link: url
Text Length: 4 characters
Text Preview: text...
Summary: summary...

Article 2: article1.json
Title: What could happen if the US strikes Iran? Here are seven scenarios
Link: https://www.bbc.com/news/articles/ce3kenge1k9o
Text Length: 9,170 characters
Text Preview: Skip to content Register Sign In What could happen if the US strikes Iran? Here are seven scenarios 6 days ago Share Save Frank Gardner Security correspondent EPA US President Donald Trump and Iran's ...
Summary: From regime change to retaliation, the BBC's Frank Gardner outlines possible outcomes of US strikes on Iran....

Article 3: article2.json
Title: What next for Venezuela? What leaders and experts said at Davos
Link: https://www.weforum.org/stories/2026/01/venezuela-what-next/
Text Length: 9,670 characters
Text Preview: GEOGRAPHIES IN DEPTH What next for Venezuela? What leaders and experts said at Davos Jan 23, 2026  Ngaire Woods: The international community has to 'creat

## Heuristic approach

In [40]:
import spacy
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class HeuristicCheckWorthy(NLPComponent):
    """
    APPROACH 1: Rule-based tagging using Spacy.
    Flags sentences containing specific entities, numbers, and verbs.
    """
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.reporting_verbs = {"say", "claim", "state", "report", "increase", "decrease", "cost", "kill"}
        
    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        for s in result.sentences:
            doc = self.nlp(s.text)
            
            has_entity = any(ent.label_ in ["PERSON", "ORG", "GPE", "LOC"] for ent in doc.ents)
            has_number = any(ent.label_ in ["MONEY", "PERCENT", "CARDINAL", "DATE"] for ent in doc.ents)
            has_verb = any(token.lemma_ in self.reporting_verbs for token in doc)
            
            # Simple scoring logic based on presence of key elements
            score = 0.0
            if has_entity: score += 0.3
            if has_number: score += 0.4
            if has_verb: score += 0.3
                
            s.is_checkworthy = bool(score > 0.6)
            s.confidence = score

## Transformer

In [41]:
from transformers import pipeline
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class TransformerCheckWorthy(NLPComponent):
    """
    APPROACH 2: Semantic classification using a Transformer model.
    Uses NLI (Natural Language Inference) to determine if a sentence constitutes a verifiable claim.
    """
    def __init__(self):
        # We use a lightweight zero-shot classifier as a proxy for a CheckThat-trained model
        self.classifier = pipeline(
            "zero-shot-classification", 
            model="cross-encoder/nli-distilroberta-base"
        )
        self.candidate_labels = ["a verifiable factual claim", "a subjective opinion or question"]
        self.threshold = 0.65

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        if not result.sentences:
            return
            
        texts = [s.text for s in result.sentences]
        
        # Batch process the sentences
        classifications = self.classifier(texts, self.candidate_labels)
        
        for i, s in enumerate(result.sentences):
            res = classifications[i]
            # Get the confidence score for the "verifiable claim" label
            claim_index = res['labels'].index("a verifiable factual claim")
            score = res['scores'][claim_index]
            
            s.is_checkworthy = bool(score >= self.threshold)
            s.confidence = score

## Prompt engineered

In [ ]:
import json
from llama_cpp import Llama
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class LLMCheckWorthy(NLPComponent):
    """
    APPROACH 3: Fast LLM Evaluation using GGUF quantized models (llama.cpp).
    Uses optimized binary format for 5-10x faster loading and 2-3x faster inference.
    """
    def __init__(self, model_path: str = "Qwen/Qwen2.5-1.5B-Instruct-GGUF", filename: str = "*q4_k_m.gguf"):
        """
        Initialize with a quantized GGUF model.
        
        Options:
        - "Qwen/Qwen2.5-1.5B-Instruct-GGUF" with "*q4_k_m.gguf" (lightweight, fast)
        - "Qwen/Qwen2.5-3B-Instruct-GGUF" with "*q4_k_m.gguf" (more capable)
        - "Qwen/Qwen2.5-7B-Instruct-GGUF" with "*q4_k_m.gguf" (best quality)
        
        Or provide local .gguf file path
        """
        print(f"Loading GGUF model from {model_path}...")
        
        # Check if it's a local file or HuggingFace repo
        if model_path.endswith('.gguf'):
            # Local file
            self.llm = Llama(
                model_path=model_path,
                n_ctx=2048,
                n_threads=4,  # Adjust based on CPU cores
                verbose=False
            )
        else:
            # Download from HuggingFace
            self.llm = Llama.from_pretrained(
                repo_id=model_path,
                filename=filename,
                n_ctx=2048,
                n_threads=4,
                verbose=False
            )
        
        print("✓ GGUF model loaded and ready")

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        # To save tokens, we only send sentences longer than 5 words
        candidates = [s for s in result.sentences if len(s.text.split()) > 5]
        
        if not candidates:
            return
        
        # Batch sentences in groups of 10 to avoid context length issues
        batch_size = 10
        
        for i in range(0, len(candidates), batch_size):
            batch = candidates[i:i+batch_size]
            sentences_json = json.dumps([s.text for s in batch], indent=2)
            
            prompt = f"""You are an expert fact-checker. For each sentence below, determine if it is "check-worthy".

A sentence is check-worthy IF it contains a factual claim AND relates to politics, health, science, or public interest where false information could cause harm.

Return ONLY a valid JSON object with a "scores" array containing numbers from 0.0 to 1.0 for each sentence.

Sentences:
{sentences_json}

Response (JSON only):"""

            try:
                # Generate response using llama.cpp
                response = self.llm.create_chat_completion(
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3,
                    max_tokens=256,
                    response_format={"type": "json_object"}
                )
                
                # Extract response text
                response_text = response['choices'][0]['message']['content'].strip()
                
                # Extract JSON from response (handles markdown code blocks)
                if "```json" in response_text:
                    response_text = response_text.split("```json")[1].split("```")[0].strip()
                elif "```" in response_text:
                    response_text = response_text.split("```")[1].split("```")[0].strip()
                
                # Parse JSON
                result_json = json.loads(response_text)
                scores = result_json.get("scores", [])
                
                # Assign scores to sentences
                for s, score in zip(batch, scores):
                    s.is_checkworthy = bool(score > 0.7)
                    s.confidence = float(score)
                    
            except (json.JSONDecodeError, KeyError, IndexError) as e:
                print(f"LLM Parsing Error: {e}")
                if 'response_text' in locals():
                    print(f"Response: {response_text[:200]}...")
                # Fallback: mark all as uncertain
                for s in batch:
                    s.is_checkworthy = False
                    s.confidence = 0.5
            except Exception as e:
                print(f"LLM Error: {e}")
                for s in batch:
                    s.is_checkworthy = False
                    s.confidence = 0.5

: 

In [ ]:
import os
import copy

# ==========================================
# 1. INITIALIZE COMPONENTS
# ==========================================
print("Initializing models... (This might take a minute for the Transformer and LLM)")

# Assuming you have your Preprocessor class defined or imported from earlier
try:
    preprocessor = Preprocessor() 
except NameError:
    print("Warning: Preprocessor not found. Make sure it's imported or defined.")

heuristic_model = HeuristicCheckWorthy()
transformer_model = TransformerCheckWorthy()

# Initialize LLM with open-source model (no API key needed)
try:
    llm_model = LLMCheckWorthy(model_name="Qwen/Qwen2.5-1.5B-Instruct")
except Exception as e:
    llm_model = None
    print(f"Notice: Could not load LLM model. Error: {e}")
    print("Skipping LLM evaluation.")

# ==========================================
# 2. EVALUATION LOOP
# ==========================================
# Test on a subset of articles to keep output readable (e.g., Article 1 and 2)
test_articles = articles[1:3] 
options = NLPOptions()

for idx, article in enumerate(test_articles):
    print(f"\n{'='*100}")
    print(f"EVALUATING ARTICLE {idx+1}: {article.title}")
    print(f"{'='*100}")
    
    # Setup base NLP Result and parse sentences
    base_result = NLPResult(sentences=[], entities_in_article=[], claims_in_article=[])
    preprocessor.run(article, base_result, options)
    
    if not base_result.sentences:
        print("No sentences found after preprocessing. Skipping.")
        continue
        
    # Take a sample of 15 sentences to keep the matrix clean
    sample_sentences = base_result.sentences[:15]
    
    # Deep copy the results so each model gets a clean slate to score
    res_heuristic = copy.deepcopy(base_result)
    res_heuristic.sentences = copy.deepcopy(sample_sentences)
    
    res_transformer = copy.deepcopy(base_result)
    res_transformer.sentences = copy.deepcopy(sample_sentences)
    
    res_llm = copy.deepcopy(base_result)
    res_llm.sentences = copy.deepcopy(sample_sentences)
    
    # --- RUN INFERENCE ---
    print("Running Heuristic Model...")
    heuristic_model.run(article, res_heuristic, options)
    
    print("Running Transformer Model...")
    transformer_model.run(article, res_transformer, options)
    
    if llm_model:
        print("Running LLM Model...")
        llm_model.run(article, res_llm, options)
    
    # --- PRINT COMPARATIVE MATRIX ---
    print(f"\n{'-'*110}")
    # Header
    print(f"{'HEURISTIC':<12} | {'TRANSFORMER':<12} | {'LLM':<12} | {'SENTENCE'}")
    print(f"{'-'*110}")
    
    for i in range(len(sample_sentences)):
        # Format: Score (T/F)
        h_flag = "T" if res_heuristic.sentences[i].is_checkworthy else "F"
        h_str = f"{res_heuristic.sentences[i].confidence:.2f} ({h_flag})"
        
        t_flag = "T" if res_transformer.sentences[i].is_checkworthy else "F"
        t_str = f"{res_transformer.sentences[i].confidence:.2f} ({t_flag})"
        
        if llm_model:
            l_flag = "T" if res_llm.sentences[i].is_checkworthy else "F"
            l_str = f"{res_llm.sentences[i].confidence:.2f} ({l_flag})"
        else:
            l_str = "N/A"
            
        # Truncate text so it fits neatly in the terminal
        text = sample_sentences[i].text.replace('\n', ' ')
        if len(text) > 65: 
            text = text[:62] + "..."
            
        print(f"{h_str:<12} | {t_str:<12} | {l_str:<12} | {text}")

microservices.nlp.components.preprocess - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


Initializing models... (This might take a minute for the Transformer and LLM)


httpx - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/nli-distilroberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/nli-distilroberta-base/b14d131f9d32668a5e6a982729b57ff6ed5dfcbd/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 299.11it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/nli-distilroberta-base/tree/main/additional_chat_templates?recursive

Loading Qwen/Qwen2.5-1.5B-Instruct... This may take a moment.


httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
